# SARA small (~45M) — Train on free Colab T4
**Runtime → Change runtime type → T4 GPU** select karo, phir upar se niche saare cells run karo.

- Clone + setup: ~2 min
- Data prep: ~1 min
- Training 5000 steps: **~1.5–2 hours**
- Output: `checkpoints/sara_small/sara.pt` (45M trained model)

In [ ]:
# 1. Clone SARA repo
!git clone https://github.com/skmandal3240/SARA
%cd SARA

In [ ]:
# 2. Install deps (Colab has torch already; add what's missing)
!pip install -q -r requirements.txt pyyaml
# quick check: model instantiates at 44.9M params on GPU
!python -c "from sara.config import SARAConfig; from sara.model import SARA; import torch; m=SARA(SARAConfig.small()); print(f'{m.n_params()/1e6:.1f}M params, cuda={torch.cuda.is_available()}')"

In [ ]:
# 3. Prepare data (tokenizer + corpus)
!python prepare_data.py --vocab 4096

In [ ]:
# 4. TRAIN — 5000 steps on T4 (~1.5-2h). Kam time chahiye to steps ghata do.
!python train_small.py --steps 5000 --batch 16

In [ ]:
# 5. Generate text from the trained model (repo's own loader)
import torch
from generate import load_sara
from pathlib import Path

model, tok, cfg = load_sara(Path('checkpoints/sara_small/sara.pt'))
prompt = tok.wrap_user('The sun is')
ids = torch.tensor([tok.encode(prompt)], dtype=torch.long)
out = model.generate(ids, max_new=40, temperature=0.8, eos_id=tok.eos_id)
print(tok.decode(out[0].tolist()))

In [ ]:
# 6. Download the trained checkpoint (45M ≈ 180 MB fp32)
from google.colab import files
files.download('checkpoints/sara_small/sara.pt')